# Truncation Blind Spot -- local GPU run notebook

Same pipeline as `colab_run.ipynb`, for a machine with its own GPU instead of Colab. No `google.colab` APIs -- Drive access here means a **locally synced Google Drive folder** (Google Drive for Desktop, or `rclone`/similar), not a mount. Point `DRIVE_ROOT` at that synced folder and everything else -- data, checkpoints, metrics -- reads/writes exactly where Colab would, so runs on this laptop and on Colab share the same experiment state (resume works across both).

Metrics still go to both W&B and Drive, same as Colab.

Prerequisites (do these once, outside this notebook, before running it):
1. A Python environment (ideally a venv **outside** this repo directory -- see `README.md`) with this kernel selected.
2. `pip install -r requirements.txt` already run in that environment, with a `torch` build matching this machine's GPU/CUDA driver (the plain `torch>=1.3` pin in `requirements.txt` doesn't guarantee the right CUDA build -- install torch yourself first per pytorch.org if `pip install -r requirements.txt` doesn't give you a CUDA-enabled build).
3. Google Drive for Desktop (or equivalent) syncing to a local folder that mirrors `MyDrive/FinalProject`.

Run the cells in order, top to bottom. The only manual edit point is `DRIVE_ROOT` in cell 2.

## 1. Point at the locally synced Drive folder

In [ ]:
import os

# Set this to wherever Google Drive for Desktop (or equivalent) syncs
# MyDrive/FinalProject to on THIS machine. Examples:
#   macOS:   "/Users/<you>/Library/CloudStorage/GoogleDrive-<email>/My Drive/FinalProject"
#   Windows: "G:/My Drive/FinalProject"
DRIVE_ROOT = "/path/to/your/synced/FinalProject"

assert os.path.isdir(DRIVE_ROOT), (
    f"Expected synced Drive folder not found: {DRIVE_ROOT}. "
    "Set DRIVE_ROOT above to the local sync path, and make sure syncing has finished."
)
print(f"Using {DRIVE_ROOT}")

## 2. Confirm dependencies (installed once, outside this notebook)

Re-running this is safe and cheap -- pip no-ops on anything already satisfied. It will NOT fix a wrong (e.g. CPU-only) torch build for you; that must be installed correctly beforehand per prerequisite 2 above.

In [ ]:
!pip install -q -r requirements.txt

## 3. Log into Weights & Biases

Prompts interactively the first time, then caches locally (no Colab secrets manager here).

In [ ]:
import wandb

wandb.login()

## 4. Environment snapshot

Confirms what this machine actually has (device, library versions) before committing GPU time to a real run.

In [ ]:
!python src/print_env.py --save {DRIVE_ROOT}/logs/env_snapshot_local_gpu.json

## 5. WikiText-103 data

Expected to already exist on the synced Drive folder (the 20,000-row cap) at `DATASET_PATH` -- same data Colab runs use. If missing, loads it fresh with the same cap.

In [ ]:
DATASET_PATH = f"{DRIVE_ROOT}/data/wikitext103"
if not os.path.exists(f"{DATASET_PATH}/train.json"):
    print(f"Not found at {DATASET_PATH}, loading fresh (max_train_samples=20000)...")
    !python src/load_data_ours.py --config-name=ours dataset.path={DATASET_PATH}
else:
    print(f"Reusing existing WikiText-103 data at {DATASET_PATH}")

## 6. Run arm(s)

Same `config/arms.yaml` manifest and `run_all_arms.py` orchestrator as Colab. Writes checkpoints/metrics under `{DRIVE_ROOT}/experiments/<arm_name>/` and logs every run to W&B.

`ARM_NAME` runs just that one arm; set `ARM_NAME = None` to run the full sweep. Safe to re-run unmodified after any interruption -- already-completed arms, iterations, and (as of the latest generate.py) even partially-generated batches within an iteration are detected and skipped, not redone.

Because this reads/writes the same synced Drive folder as `colab_run.ipynb`, you can freely split arms between this laptop and Colab, or resume on whichever machine is available -- both see the same completed/in-progress state.

In [ ]:
ARM_NAME = "minp_005"  # set to None to run the full sweep

arm_flag = f"--arms {ARM_NAME}" if ARM_NAME else ""
!DRIVE_ROOT={DRIVE_ROOT} python run_all_arms.py --drive-root {DRIVE_ROOT} {arm_flag}